<a href="https://colab.research.google.com/github/HelloPenguin1/Spine_Lesion_Classification_Using_BasicML/blob/main/TCIA_CT_Preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 14.5 MB/s eta 0:00:00


In [17]:
from pathlib import Path
import SimpleITK as sitk
import numpy as np

Loading Data

In [12]:
def load_patient_images(ct_path, seg_path):
    ct_img = sitk.ReadImage(str(ct_path))
    seg_img = sitk.ReadImage(str(seg_path))

    return ct_img, seg_img

Geometry Alignment

In [13]:
def align_seg_to_ct(ct_img, seg_img):

    seg_resampled = sitk.Resample(
        seg_img,
        ct_img,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        seg_img.GetPixelID()
    )

    return seg_resampled

Convert to Numpy

In [14]:
def images_to_numpy(ct_img, seg_img):

    ct_np = sitk.GetArrayFromImage(ct_img)
    seg_np = sitk.GetArrayFromImage(seg_img)

    return ct_np, seg_np

Vertebrae Extraction

In [15]:
def extract_vertebra_mask(seg_np, vertebra_label):

    mask = seg_np == vertebra_label

    return mask

Bounding Box Extraction

In [16]:
def compute_bbox(mask):

    coords = np.where(mask)

    zmin, zmax = coords[0].min(), coords[0].max()
    ymin, ymax = coords[1].min(), coords[1].max()
    xmin, xmax = coords[2].min(), coords[2].max()

    return (
        zmin, zmax,
        ymin, ymax,
        xmin, xmax
    )

Margin Expansion: to preserve near anatomical features

In [18]:
def add_margin(
    bbox,
    volume_shape,
    margin_x=10,
    margin_y=10,
    margin_z=5
):

    zmin,zmax,ymin,ymax,xmin,xmax = bbox

    xmin = max(0, xmin - margin_x)
    xmax = min(volume_shape[2], xmax + margin_x)

    ymin = max(0, ymin - margin_y)
    ymax = min(volume_shape[1], ymax + margin_y)

    zmin = max(0, zmin - margin_z)
    zmax = min(volume_shape[0], zmax + margin_z)

    return (
        zmin,zmax,
        ymin,ymax,
        xmin,xmax
    )

CT Cropping: remove irrelevant anatomy

In [19]:
def crop_volume(volume, bbox):

    zmin,zmax,ymin,ymax,xmin,xmax = bbox

    crop = volume[
        zmin:zmax+1,
        ymin:ymax+1,
        xmin:xmax+1
    ]

    return crop

Isotropic Resampling: standardize voxel size


In [23]:
import SimpleITK as sitk


def resample_to_isotropic(ct_crop_img, target_spacing=(1.0, 1.0, 1.0)):
    """
    Resample a cropped CT image to isotropic voxel spacing.

    Parameters
    ----------
    ct_crop_img : SimpleITK.Image
        Cropped vertebral CT image.

    target_spacing : tuple
        Desired voxel spacing (default = 1x1x1 mm).

    Returns
    -------
    ct_iso_img : SimpleITK.Image
        Isotropically resampled CT image.
    """

    old_spacing = ct_crop_img.GetSpacing()
    old_size = ct_crop_img.GetSize()

    new_size = [
        int(round(old_size[i] * old_spacing[i] / target_spacing[i]))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()

    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)

    resampler.SetOutputDirection(
        ct_crop_img.GetDirection()
    )

    resampler.SetOutputOrigin(
        ct_crop_img.GetOrigin()
    )

    resampler.SetInterpolator(
        sitk.sitkLinear
    )

    ct_iso_img = resampler.Execute(ct_crop_img)

    return ct_iso_img

HU Windowing: suppress soft tissue

In [26]:
def apply_bone_window(
    volume,
    hu_min=-450,
    hu_max=1050
):
    return np.clip(
        volume,
        hu_min,
        hu_max
    )

Normalize Intensity: for stabilized input for Neural Network

In [22]:
def normalize_intensity(
    volume,
    hu_min=-450,
    hu_max=1050
):

    volume = (
        volume - hu_min
    ) / (
        hu_max - hu_min
    )

    volume = (
        volume * 2
    ) - 1

    return volume

Resize to Network Input

In [27]:
import numpy as np
import SimpleITK as sitk


def resize_to_network_input(
    volume,
    reference_spacing=(1.0, 1.0, 1.0),
    target_size=(64, 64, 64)
):
    """
    Resize a normalized isotropic CT volume to the
    fixed network input size.

    Parameters
    ----------
    volume : numpy.ndarray
        Normalized isotropic CT volume.

    reference_spacing : tuple
        Current voxel spacing.
        After isotropic resampling this is (1,1,1).

    target_size : tuple
        Desired network input size.

    Returns
    -------
    numpy.ndarray
        Volume of shape (64,64,64)
    """

    img = sitk.GetImageFromArray(
        volume.astype(np.float32)
    )

    img.SetSpacing(reference_spacing)

    resampler = sitk.ResampleImageFilter()

    resampler.SetSize(target_size)

    new_spacing = [
        img.GetSize()[i]
        * img.GetSpacing()[i]
        / target_size[i]
        for i in range(3)
    ]

    resampler.SetOutputSpacing(new_spacing)

    resampler.SetInterpolator(
        sitk.sitkLinear
    )

    img_resized = resampler.Execute(img)

    volume_resized = sitk.GetArrayFromImage(
        img_resized
    )

    return volume_resized